# Non-performing load (NPL) - Data Cleaning

### Data Info:
* **Indicator:** `частка непрацюючих кредитів, % (NPL)`
* **Source:** National Bank of Ukraine: https://bank.gov.ua/ua/statistic/supervision-statist
* **Raw File:** `data_raw/NPL.xlsx`
* **Output:** `data_processed/NPL_clean.xlsx`  

### Cleaning Notes:
- wide format: rows = indicators, columns = dates
- date row contains footnote marker (`**`)

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

In [2]:
file_path = Path("../../data_raw/NPL.xlsx")
output_path = Path("../../data_processed/NPL_clean.xlsx")
output_path.parent.mkdir(parents=True, exist_ok=True)

sheet_name = "NPL_Total"

df_raw = pd.read_excel(file_path, sheet_name=sheet_name, header=None)
df_raw.head()

,0,1,2,3,4,5,6,7,8,9,...,101,102,103,104,105,106,107,108,109,110
0,Обсяги активних операцій та частка непрацюючих...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,млн.грн.
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Активна операція,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,2/1/2017**,2017-03-01 00:00:00,2017-04-01 00:00:00,2017-05-01 00:00:00,2017-06-01 00:00:00,2017-07-01 00:00:00,2017-08-01 00:00:00,2017-09-01 00:00:00,2017-10-01 00:00:00,...,2025-06-01 00:00:00,2025-07-01 00:00:00,2025-08-01 00:00:00,2025-09-01 00:00:00,2025-10-01 00:00:00,2025-11-01 00:00:00,2025-12-01 00:00:00,2026-01-01 00:00:00,2026-02-01 00:00:00,2026-03-01 00:00:00
4,Кредитні операції:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
# set correct header

header_row_idx = 3          # Date: excel row 4
target_row_idx = 33         # NPL: excel row 34

header_values = df_raw.iloc[header_row_idx].copy()
header_values.iloc[0] = "indicator"

df = df_raw.iloc[[target_row_idx]].copy()
df.columns = header_values
df = df.reset_index(drop=True)
display(df.head())

3,indicator,2/1/2017**,2017-03-01 00:00:00,2017-04-01 00:00:00,2017-05-01 00:00:00,2017-06-01 00:00:00,2017-07-01 00:00:00,2017-08-01 00:00:00,2017-09-01 00:00:00,2017-10-01 00:00:00,...,2025-06-01 00:00:00,2025-07-01 00:00:00,2025-08-01 00:00:00,2025-09-01 00:00:00,2025-10-01 00:00:00,2025-11-01 00:00:00,2025-12-01 00:00:00,2026-01-01 00:00:00,2026-02-01 00:00:00,2026-03-01 00:00:00
0,"частка непрацюючих кредитів, %",53.989765,55.899839,55.106022,56.597416,56.391397,57.732746,57.992259,56.93538,56.435632,...,27.890702,27.014141,26.138961,25.345659,25.035591,24.462379,23.913705,13.922485,13.865838,13.341712


In [4]:
# clean 

def clean_date_string(x):
    if pd.isna(x):
        return np.nan
    
    x = str(x).strip()
    
    x = re.sub(r"\*+", "", x)
    x = re.sub(r"#", "", x)
    x = re.sub(r"\s*\(.*?\)", "", x)
    x = x.strip()
    
    return x

new_columns = []
for col in df.columns:
    if col in ["indicator"]:
        new_columns.append(col)
    else:
        cleaned = clean_date_string(col)
        new_columns.append(cleaned)

df.columns = new_columns

df.iloc[:5, :8]

,indicator,2/1/2017,2017-03-01 00:00:00,2017-04-01 00:00:00,2017-05-01 00:00:00,2017-06-01 00:00:00,2017-07-01 00:00:00,2017-08-01 00:00:00
0,"частка непрацюючих кредитів, %",53.989765,55.899839,55.106022,56.597416,56.391397,57.732746,57.992259


In [5]:
print(df.columns)
print(type(df.columns[1]))

Index(['indicator', '2/1/2017', '2017-03-01 00:00:00', '2017-04-01 00:00:00',
       '2017-05-01 00:00:00', '2017-06-01 00:00:00', '2017-07-01 00:00:00',
       '2017-08-01 00:00:00', '2017-09-01 00:00:00', '2017-10-01 00:00:00',
       ...
       '2025-06-01 00:00:00', '2025-07-01 00:00:00', '2025-08-01 00:00:00',
       '2025-09-01 00:00:00', '2025-10-01 00:00:00', '2025-11-01 00:00:00',
       '2025-12-01 00:00:00', '2026-01-01 00:00:00', '2026-02-01 00:00:00',
       '2026-03-01 00:00:00'],
      dtype='object', length=111)
<class 'str'>


In [6]:
# clean date columns
df = df.rename(columns={
    "2/1/2017": "2017-02-01"
})

new_columns = []

for col in df.columns:
    if col == "indicator":
        new_columns.append(col)
    else:
        col = str(col).strip()
        col = col.replace(" 00:00:00", "")
        new_columns.append(col)

df.columns = new_columns

print(df.columns)

Index(['indicator', '2017-02-01', '2017-03-01', '2017-04-01', '2017-05-01',
       '2017-06-01', '2017-07-01', '2017-08-01', '2017-09-01', '2017-10-01',
       ...
       '2025-06-01', '2025-07-01', '2025-08-01', '2025-09-01', '2025-10-01',
       '2025-11-01', '2025-12-01', '2026-01-01', '2026-02-01', '2026-03-01'],
      dtype='object', length=111)


In [7]:
# transform to long format (melt)

date_columns = [col for col in df.columns if col != "indicator"]

df_long = df.melt(
    id_vars="indicator",
    value_vars=date_columns,
    var_name="date",
    value_name="value"
).copy()

df_long["date"] = pd.to_datetime(df_long["date"], errors="coerce")
df_long["indicator"] = "NPL"

df_long

,indicator,date,value
0,NPL,2017-02-01,53.989765
1,NPL,2017-03-01,55.899839
2,NPL,2017-04-01,55.106022
3,NPL,2017-05-01,56.597416
4,NPL,2017-06-01,56.391397
...,...,...,...
105,NPL,2025-11-01,24.462379
106,NPL,2025-12-01,23.913705
107,NPL,2026-01-01,13.922485
108,NPL,2026-02-01,13.865838


In [8]:
# convert values to numeric

df_long["value"] = pd.to_numeric(df_long["value"], errors="coerce")

df_final = df_long[["date", "value", "indicator"]].copy()
df_final = df_final.dropna(subset=["date", "value"]).sort_values("date").reset_index(drop=True)

df_final.head()

,date,value,indicator
0,2017-02-01,53.989765,NPL
1,2017-03-01,55.899839,NPL
2,2017-04-01,55.106022,NPL
3,2017-05-01,56.597416,NPL
4,2017-06-01,56.391397,NPL


In [9]:
# validate 

df_final["date"] = pd.to_datetime(df_final["date"]).dt.date

print(df_final.dtypes)
print("\nmissing values:")
print(df_final.isna().sum())

date          object
value        float64
indicator     object
dtype: object

missing values:
date         0
value        0
indicator    0
dtype: int64


In [10]:
df_final

,date,value,indicator
0,2017-02-01,53.989765,NPL
1,2017-03-01,55.899839,NPL
2,2017-04-01,55.106022,NPL
3,2017-05-01,56.597416,NPL
4,2017-06-01,56.391397,NPL
...,...,...,...
105,2025-11-01,24.462379,NPL
106,2025-12-01,23.913705,NPL
107,2026-01-01,13.922485,NPL
108,2026-02-01,13.865838,NPL


In [11]:
df_final.to_excel(output_path, index=False)
print(f"saved to: {output_path}")

saved to: ..\..\data_processed\NPL_clean.xlsx
